# 35 · Stream-stream joins with watermarks and time bounds

**Scenario:** Join pickup and dropoff event views derived from the same real trips.

**Source:** Databricks-provided `samples.nyctaxi.trips` only. File formats, nested structures, replays and histories in this lab are derived from that asset. No hand-created record dataset, third-party download, pandas or generated replacement data is used.

**Prerequisites:** Run notebook 00 first. Each notebook is independent; run its cells in order. Mode: **streaming**. 

**Learning-size sample:** `max_rows` defaults to 2,000. Sorting/limiting caps result size, not the underlying scan cost. Assertions use relationships instead of hardcoded dataset counts because sample assets may change.


## Step 1 — Load and inspect the Databricks asset

Read the built-in sample and normalize column names. A stable content fingerprint helps with replay examples; it is not a guaranteed real-world trip key. The original `samples` asset is always read-only.


In [ ]:
from pyspark.sql import functions as F, types as T, Window
dbutils.widgets.text('max_rows','2000')
MAX_ROWS=int(dbutils.widgets.get('max_rows'))
assert 100<=MAX_ROWS<=20000, 'Use 100–20000 rows for these bounded learning labs.'
SOURCE_ASSET='samples.nyctaxi.trips'
try:
    asset=(spark.read.table(SOURCE_ASSET)
        .select('tpep_pickup_datetime','tpep_dropoff_datetime','trip_distance',
                'fare_amount','pickup_zip','dropoff_zip')
        .orderBy('tpep_pickup_datetime','tpep_dropoff_datetime','pickup_zip',
                 'dropoff_zip','trip_distance','fare_amount')
        .limit(MAX_ROWS))
    assert asset.limit(1).count()==1, 'Databricks sample asset is empty.'
except Exception as exc:
    raise RuntimeError('This course requires the Databricks-provided samples.nyctaxi.trips asset. '
        'Check Unity Catalog/sample access on the attached compute. No synthetic or external fallback is used.') from exc
df=(asset.select(F.col('tpep_pickup_datetime').cast('timestamp').alias('pickup_ts'),
    F.col('tpep_dropoff_datetime').cast('timestamp').alias('dropoff_ts'),
    F.col('trip_distance').cast('double').alias('distance'),
    F.col('fare_amount').cast('double').alias('fare'),
    F.col('pickup_zip').cast('string').alias('pickup_zip'),
    F.col('dropoff_zip').cast('string').alias('dropoff_zip')))
df=df.withColumn('trip_id',F.sha2(F.to_json(F.struct(*[F.col(c) for c in df.columns]),
    options={'ignoreNullFields':'false'}),256))
df=df.withColumn('pickup_date',F.to_date('pickup_ts'))
df=df.withColumn('duration_sec',F.unix_timestamp('dropoff_ts')-F.unix_timestamp('pickup_ts'))
print('Source:',SOURCE_ASSET,'sample limit:',MAX_ROWS)
# trip_id is a content fingerprint, NOT an official source/business trip identifier.
# Identical source rows intentionally have identical fingerprints.
df.show(5,truncate=False)


## Workspace setup — Choose course-owned output names

Set `target_catalog` to a catalog where you can create a schema. This lab creates a per-user course schema and uses lab-specific names. Rerunning a write lab can replace its own course tables. Nothing is written to the `samples` catalog.


In [ ]:
LAB='35'

import hashlib, re
dbutils.widgets.text('target_catalog','')
target_catalog=dbutils.widgets.get('target_catalog').strip() or spark.sql('SELECT current_catalog()').first()[0]
assert target_catalog!='samples', 'Set target_catalog to a writable catalog, not the read-only samples catalog.'
user=spark.sql('SELECT current_user()').first()[0]
schema='pyspark_lab_'+hashlib.sha256(user.encode()).hexdigest()[:10]
def quote_identifier(value): return '`'+value.replace('`','``')+'`'
namespace=f'{quote_identifier(target_catalog)}.{quote_identifier(schema)}'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {namespace}')
def table(name):
    assert re.fullmatch(r'[a-z][a-z0-9_]*',name)
    return f'{namespace}.`lab_{LAB}_{name}`'
print('Course-only namespace:',namespace)

import uuid
assert all(re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*',s) for s in [target_catalog,schema]), 'Use simple catalog/schema names for volume paths.'
spark.sql(f'CREATE VOLUME IF NOT EXISTS {namespace}.`course_files`')
run_token=uuid.uuid4().hex
base=f'/Volumes/{target_catalog}/{schema}/course_files/{LAB}/{run_token}'
print('This run writes only beneath:',base)


## Step 2 — Derive two streams from asset timestamps

Each side has its own event time. Only valid trips with a duration up to one day are included.


In [ ]:
valid=df.filter('pickup_ts IS NOT NULL AND dropoff_ts IS NOT NULL AND duration_sec >= 0 AND duration_sec <= 86400').dropDuplicates(['trip_id']).limit(200)
assert valid.limit(1).count()==1, 'Increase max_rows if no valid trips exist in the slice.'
left_table=table('pickups_'+run_token); right_table=table('dropoffs_'+run_token); sink=table('joined_'+run_token)
valid.select('trip_id',F.col('pickup_ts').alias('pickup_event_time'),'pickup_zip').write.format('delta').mode('errorifexists').saveAsTable(left_table)
valid.select('trip_id',F.col('dropoff_ts').alias('dropoff_event_time'),'dropoff_zip').write.format('delta').mode('errorifexists').saveAsTable(right_table)
left=spark.readStream.table(left_table).withWatermark('pickup_event_time','2 days').alias('p')
right=spark.readStream.table(right_table).withWatermark('dropoff_event_time','2 days').alias('d')


**Expected check:** Both streams contain only projections of the source trip records.


## Step 3 — Bound the join state

A key equality alone can retain state indefinitely. A time interval plus watermarks allows state eviction.


In [ ]:
joined=left.join(right,F.expr('p.trip_id=d.trip_id AND d.dropoff_event_time >= p.pickup_event_time AND d.dropoff_event_time <= p.pickup_event_time + INTERVAL 1 DAY'),'inner')
joined=joined.select(F.col('p.trip_id').alias('trip_id'),'pickup_event_time','dropoff_event_time','pickup_zip','dropoff_zip')
q=(joined.writeStream.format('delta').outputMode('append').option('checkpointLocation',base+'/checkpoint')
    .trigger(availableNow=True).toTable(sink))
q.awaitTermination()
if q.exception() is not None: raise RuntimeError(str(q.exception()))
out=spark.table(sink)
assert out.count()<=valid.count()
assert out.filter('dropoff_event_time < pickup_event_time').count()==0
print('Joined rows:',out.count(),'candidate trips:',valid.count())
print(q.lastProgress)


**Expected check:** Matched rows obey the one-day interval. Late-data/micro-batch behavior can reduce output; inspect progress rather than assuming universal arrival-order equivalence.


## Practice

Compare a stream-static join for the ZIP dimension. Which side needs state and why?

## Solution guide

A stream-static dimension lookup does not keep two unbounded event histories. A stream-stream join needs retained state on the input sides until no future timely match is possible.

## Production considerations

Keep source lineage and stable business keys. Measure on representative volumes, review nulls and time semantics, and make retries idempotent. Sample-data checks demonstrate mechanics; they do not establish production performance or exactly-once external writes.
